John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [ ]:
# Import packages

library(tidyverse)

### Load and Trim Dataset

In [ ]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 11694 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [ ]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [ ]:
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    for (d in unique(na.omit(filter(data, ID==ids[[i]])$date))) {
        print(d)
        print(nrow(filter(data, ID == ids[[i]], date == d)))
    }
}

[1] "LbOW"
[1] 19125
[1] 80
[1] 19126
[1] 170
[1] 19128
[1] 5
[1] 19129
[1] 230
[1] 19130
[1] 224
[1] "RbDgY"
[1] 19128
[1] 214
[1] 19129
[1] 224
[1] 19130
[1] 221
[1] "YKY"
[1] 19125
[1] 172
[1] 19126
[1] 191
[1] 19128
[1] 259
[1] 19129
[1] 238
[1] 19130
[1] 212
[1] "TbTbLg"
[1] 19125
[1] 169
[1] 19126
[1] 169
[1] 19128
[1] 164
[1] 19129
[1] 112
[1] 19130
[1] 142
[1] "DgTbO"
[1] 19125
[1] 202
[1] 19126
[1] 233
[1] 19128
[1] 241
[1] 19129
[1] 260
[1] 19130
[1] 233
[1] "RRO"
[1] 19125
[1] 65
[1] 19126
[1] 158
[1] 19128
[1] 136
[1] 19129
[1] 125
[1] 19130
[1] 54


In [ ]:
print(unique(na.omit(data$ID)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [ ]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>%
    arrange(DateTime) %>%
    mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
    group_by(date) %>%
    fill({time_last_song}, .direction = "down") %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
    ungroup()
}

[1] "LbOW"
[1] "time_M1_last_song"
[1] "RbDgY"
[1] "time_M2_last_song"
[1] "YKY"
[1] "time_M3_last_song"
[1] "TbTbLg"
[1] "time_M4_last_song"
[1] "DgTbO"
[1] "time_M5_last_song"
[1] "RRO"
[1] "time_M6_last_song"


In [ ]:
slice_sample(focal_match_data, n=5)

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,⋯,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song
<chr>,<dttm>,<date>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DgTbO,2022-05-13 05:37:46,2022-05-13,-1123.6317,7,DgTbO_2,NA,46.47336,67.67739,62.95599,⋯,NA,NA,-1127.4247,3.7929560,-1125.1701,1.538374,-1123.6317,0.000000,-1341.9084,218.276697
LbOW,2022-05-17 05:39:31,2022-05-17,-928.7465,3,LbOW_5,94.03680,39.73681,67.67739,60.45910,⋯,-929.9379,1.191393,-933.3447,4.5981552,-930.1219,1.375416,-932.5954,3.848942,-930.0887,1.342171
DgTbO,2022-05-16 05:38:35,2022-05-16,-1014.7119,33,DgTbO_5,78.37111,49.01987,67.67739,62.95599,⋯,-1015.8914,1.179481,-1015.1601,0.4482609,-1016.3784,1.666568,-1014.7119,0.000000,-1025.4387,10.726779
LbOW,2022-05-13 05:50:47,2022-05-13,-342.8622,8,LbOW_4,NA,54.16852,67.67739,62.95599,⋯,NA,NA,-350.2313,7.3691639,-352.3636,9.501451,-398.7134,55.851203,-399.1758,56.313648
RbDgY,2022-05-18 05:35:03,2022-05-18,-1196.1511,18,RbDgY_11,94.03680,39.73681,67.67739,62.95599,⋯,-1196.1511,0.000000,-1201.0612,4.9101126,-1197.2060,1.054947,-1201.0730,4.921969,-1198.6746,2.523533


In [ ]:
# Focal Male's song-type repertoire
focal_repertoire <- unique(filter(data, ID == focal_id)$UST)
print("Focal Male's Song Type Repertoire:")
focal_repertoire

# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Create column for M1's last UST
    mutate(M1_last_UST = lag(UST)) %>%
# Create column for time of and time since M1's last UST    
    mutate(time_M1_last_song = lag(time_rel_sunrise)) %>%
    mutate(time_since_M1_last_song = time_rel_sunrise - time_M1_last_song) %>%
# Input variables: M1-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        UST,
        "M1_last_UST", "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
    	time_since_M1_last_song, time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
        number_of_matches, does_it_match,
        DateTime, time_rel_sunrise
        # Also consider adding "streak"
        # Also consider adding MX nth last song
    ) %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) %>%
# Set all neighbours' songs that aren't in M1's repertoire to 0
    mutate(across(contains("UST"), ~ if_else (.x %in% focal_repertoire, .x, as.character(0))))

[1] "Focal Male's Song Type Repertoire:"


[1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12

In [ ]:
# Compared to M1's repertoire, which songs are in the other males' repertoires?
print("Male 1's Repertoire:")
print(focal_repertoire)
for (i in 2:6) {
    print(sprintf("Male %i's Repertoire:", i))
    print(pull(unique(focal_match_data %>% select({sprintf("M%i_last_UST", i)}))))
    # print("\n")
}


[1] "Male 1's Repertoire:"
 [1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12
[1] "Male 2's Repertoire:"
 [1] NA   "0"  "8"  "9"  "3"  "21" "13" "5"  "16" "20" "17"
[1] "Male 3's Repertoire:"
 [1] "0"  "10" "3"  "8"  "13" "16" NA   "17" "20" "9"  "5"  "21"
[1] "Male 4's Repertoire:"
 [1] "13" "0"  "10" "9"  "8"  "3"  "16" "21" "5"  "20" NA   "17"
[1] "Male 5's Repertoire:"
[1] "20" "0"  "10" "8"  "12" NA  
[1] "Male 6's Repertoire:"
 [1] "0"  "12" "8"  "3"  "9"  "13" "10" NA   "17" "5" 


In [ ]:
head(focal_match_data)

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M1_last_UST,M2_last_UST,M3_last_UST,M4_last_UST,⋯,time_since_M1_last_song,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song,number_of_matches,does_it_match,DateTime,time_rel_sunrise
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<fct>,<dttm>,<dbl>
NA,52.93973,67.67739,62.95599,53.6376,10,NA,NA,0,13,⋯,NA,NA,5.472131,5.626019,4.7073998,180.3157,0,0,2022-05-13 05:37:08,-1161.593
NA,46.47336,67.67739,62.95599,53.6376,10,10,NA,10,0,⋯,12.808312,NA,1.149475,4.305930,-0.2417286,193.1240,1,1,2022-05-13 05:37:21,-1148.784
NA,46.47336,67.67739,62.95599,53.6376,3,10,NA,10,10,⋯,8.334139,NA,9.483614,8.158114,1.1915795,201.4581,0,0,2022-05-13 05:37:29,-1140.450
NA,46.47336,67.67739,62.95599,53.6376,8,3,NA,3,9,⋯,9.867840,NA,2.253478,1.129353,5.5875481,211.3260,0,0,2022-05-13 05:37:39,-1130.582
NA,46.47336,67.67739,62.95599,53.6376,9,8,NA,8,0,⋯,8.200337,NA,5.042585,2.788003,1.2496293,219.5263,0,0,2022-05-13 05:37:47,-1122.382
NA,46.47336,67.67739,62.95599,53.6376,13,9,NA,0,8,⋯,14.084462,NA,2.726158,8.201041,3.7281486,233.6108,0,0,2022-05-13 05:38:01,-1108.298


### Export Data

In [ ]:
write_csv(focal_match_data, "../data/matching_opportunities.csv")